## Size of each loyalty program

In [ ]:
sns.set_style("white") 
plt.figure(figsize=(8,6))

custom_order = ["Star", "Nova", "Aurora"] # Order them as given in the report Star > Nova > Aurora

# Convert to categorical with new order
df_customer_copy["LoyaltyStatus"] = pd.Categorical(
    df_customer_copy["LoyaltyStatus"], 
    categories=custom_order, 
    ordered=True
)

# Start plot
ax = sns.countplot(data=df_customer_copy, x="LoyaltyStatus", hue="LoyaltyStatus",
              order=custom_order, hue_order=custom_order)

# Add count labels on top of bars
for container in ax.containers:
    ax.bar_label(container, fmt='%d')

# Styling for the poster
plt.title("Size of each Loyalty Program", size=25, fontweight='bold')
plt.grid(axis='y', alpha=1, linestyle='--')
plt.xlabel('Loyalty Program', fontsize=20, fontweight='bold')
plt.ylabel('Count', fontsize=20, fontweight='bold')

plt.tight_layout()
# Save as scalable vector graphic - good resolution
#plt.savefig("./Figures/Loyalty_program_size.svg", format="svg")
plt.show()

## Distribution of Flights in Loyalty Programs

In [ ]:
sns.set_style("white") 
plt.figure(figsize=(8,6))

# Start plotting
sns.histplot(df_customer_copy, x = "Flights_in_Subscription", hue = "LoyaltyStatus", bins=10, multiple="stack", binwidth=50, alpha=1)

# Styling
plt.title("Distribution of Flights in Loyalty Programs", size=25, fontweight='bold')
plt.grid(axis='y', alpha=1, linestyle='--')
plt.xlabel('Flights during Subscription', fontsize=20, fontweight='bold')
plt.ylabel('Count', fontsize=20, fontweight='bold')

plt.tight_layout()
# Save as svg
#plt.savefig("./Figures/Flights_in_Subscription.svg", format="svg")
plt.show()

## Income Distribution in each Loyalty Program

In [ ]:
sns.set_style("white") 
plt.figure(figsize=(8,6))

# Start plotting
sns.histplot(df_customer_copy, x="Income", hue="LoyaltyStatus", 
             binwidth=10000, multiple="stack", alpha=1) # binwidth=10000 since we have a max income of 100000

# Styling
plt.title("Income Distribution in each Loyalty Progam", size=25, fontweight='bold')
plt.grid(axis='y', alpha=1, linestyle='--')
plt.xlabel('Customer Income', fontsize=20, fontweight='bold')
plt.ylabel('Count', fontsize=20, fontweight='bold')
plt.xticks([10000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000]) # Define the x-axis tick values so they are clearly readable
plt.tight_layout()

# Save plot as svg
#plt.savefig("./Figures/Income_loyalty.svg", format="svg")
plt.show()

## Travel km per Loyalty Program

In [ ]:
sns.set_style("white") 
plt.figure(figsize=(8,6))

sns.histplot(df_customer_copy, x = "Total_Distance_KM", hue = "LoyaltyStatus", binwidth=100000, multiple="stack", alpha=1)

# Styling
plt.title("Travel km for each Loyalty Program", size=25, fontweight='bold')
plt.grid(axis='y', alpha=1, linestyle='--')
plt.xlabel('Distance (km)', fontsize=20, fontweight='bold')
plt.ylabel('Count', fontsize=20, fontweight='bold')
plt.tight_layout()

# Save plot
#plt.savefig("./Figures/travel_loayalty.svg", format="svg")
plt.show()

## Loyalty Program Membership Growth

In [ ]:
sns.set_style("white")

# Prepare data
# Change date column to datetime
df_customer_copy['EnrollmentDateOpening'] = pd.to_datetime(df_customer_copy['EnrollmentDateOpening'])
df_customer_copy['CancellationDate'] = pd.to_datetime(df_customer_copy['CancellationDate'])

# Create date range
year_months = df_flights[['Year', 'Month']].drop_duplicates().sort_values(['Year', 'Month']) # Create a Dataframe with Year and Month columns

# First we create a new column day which we assing 1, then we add the monthEnd day to this column (e.g. 31 for January) and finally we convert this to a datetime and assign it to the Date column
year_months['Date'] = pd.to_datetime(year_months[['Year', 'Month']].assign(day=1)) + pd.offsets.MonthEnd(0) 

# Count active customers
results = []
# Create a subset dataframe with only customer that where active within the date
for date in year_months['Date']: 
    active = df_customer_copy[
        (df_customer_copy['EnrollmentDateOpening'] <= date) & 
        (df_customer_copy['CancellationDate'] > date)
    ]
    # Create the final dataframe with the date (e.g 2020-01-31) and the number of active customers for each loyalty program
    for status, count in active['LoyaltyStatus'].value_counts().items():
        results.append({'Date': date, 'LoyaltyStatus': status, 'NumCustomers': count})

# Turn to dataframe
program_growth = pd.DataFrame(results)

# Pivot the dataframe so we get Loayalty Program as columns and Date as index
program_pivot = program_growth.pivot(index='Date', columns='LoyaltyStatus', values='NumCustomers')
program_pivot = program_pivot[["Star", "Nova", "Aurora"]] # Change order to Star > Nova > Aurora

# Start plotting
# Stacked area chart 
# Note use matplot lib as the graph was not available in seaborn (to my knowledge)
fig, ax = plt.subplots(figsize=(11, 6))
program_pivot.plot.area(stacked=True, alpha=0.8, ax=ax)

# Styling for Poster
plt.title('Loyalty Program Membership Growth', fontsize=25, fontweight='bold')
plt.xlabel('Date', fontsize=20, fontweight='bold')
plt.ylabel('Active Customers', fontsize=20, fontweight='bold')
plt.grid(axis="y", alpha=1, linestyle='--')
plt.grid(axis='x', alpha=1, linestyle='--')
plt.legend(title='Loyalty Status', fontsize=11)
plt.xticks()
plt.tight_layout()

# Save as svg
#plt.savefig("./Figures/program_growth.svg", format="svg")
plt.show()

## Flights booked per city over the years 2020-2021

In [ ]:
# Data preparation
df_customer_copy["Enrollment_Year"] = df_customer_copy["EnrollmentDateOpening"].dt.year # Extract year from date

# Create dataframe with Enrollment Year, Loayalty Status, mean Customer Lifetime Value
clv_by_year = df_customer_copy.groupby(['Enrollment_Year', 'LoyaltyStatus'])["Customer Lifetime Value"].mean().sort_values().reset_index()

# Change loaylty program to categorical variable
# Assigned order Star > Nova > Aurora
clv_by_year['LoyaltyStatus'] = pd.Categorical(clv_by_year['LoyaltyStatus'], 
                                            categories=["Star", "Nova", "Aurora"], 
                                            ordered=True)

# Calculate indexed values
baseline_year = clv_by_year['Enrollment_Year'].min()
baseline_vals = clv_by_year[clv_by_year['Enrollment_Year'] == baseline_year].set_index('LoyaltyStatus')['Customer Lifetime Value'] # reindex starting with 2015

# Calculate a new column which shows the percentage change in CLV over the years
clv_by_year['Change_CLV'] = clv_by_year.apply(
    lambda x: (x['Customer Lifetime Value'] / baseline_vals[x['LoyaltyStatus']]) * 100, # for each row select Customer Lifetime Value and divide by the baseline value for the same LoyaltyStatus
    axis=1
).round(3)


# define subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
sns.set_style("white") 

# Plot 1: Absolute Value development
sns.lineplot(data=clv_by_year, x='Enrollment_Year', y='Customer Lifetime Value', 
             hue='LoyaltyStatus', style='LoyaltyStatus',
             markers={'Star': 'o', 'Nova': 's', 'Aurora': '^'}, 
             markersize=10, linewidth=3, ax=ax1)

# Styling
ax1.set_ylim(0, None)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax1.set_xlabel('Enrollment Year', fontsize=18, fontweight='bold')
ax1.set_ylabel('Average CLV', fontsize=18, fontweight='bold')
ax1.set_title('Absolute Customer Lifetime Values for each Loyalty Program', fontsize=16, fontweight='bold')
ax1.legend(title='Loyalty Status', fontsize=10, loc='lower right')
ax1.grid(axis='y', alpha=1, linestyle='--')


# Plot 2: Relative Value development
sns.lineplot(data=clv_by_year, x='Enrollment_Year', y='Change_CLV', 
             hue='LoyaltyStatus', style='LoyaltyStatus',
             markers={'Star': 'o', 'Nova': 's', 'Aurora': '^'}, 
             markersize=10, linewidth=3, ax=ax2)

# Styling
ax2.axhline(y=100, color='gray', linestyle=':', linewidth=2, alpha=0.7)
ax2.set_xlabel('Enrollment Year', fontsize=18, fontweight='bold')
ax2.set_ylabel("Relative CLV change in % since 2015", fontsize=18, fontweight='bold')
ax2.set_title('Relative Change in Customer Lifetime Values since 2015', fontsize=16, fontweight='bold')
ax2.legend(title='Loyalty Status', fontsize=10, loc='lower left')
ax2.grid(axis='y', alpha=1, linestyle='--')


plt.suptitle('Customer Lifetime Value (CLV) Analysis by Loyalty Program', 
             fontsize=25, fontweight='bold')
plt.tight_layout()

# Save figure as svg
#plt.savefig("./Figures/CLV_analysis.svg", format="svg")
plt.show()

## Flight Distribution

In [ ]:
# Filter for 2021 data
df_merged_2021 = df_merged[df_merged['Year'] == 2021]

# Calculate total flights for each city in 2021
flights_2021 = df_merged_2021.groupby('City')['NumFlights'].sum()

# Define major cities
major_cities = ['Toronto', 'Vancouver', 'Montreal']

# Calculate flights for major cities and others
major_flights = flights_2021[flights_2021.index.isin(major_cities)].sum()
other_flights = flights_2021[~flights_2021.index.isin(major_cities)].sum()


labels = ['Toronto,\n Vancouver\n& Montreal', 'Other\n Canadian\nCities']
sizes = [47.6, 52.4]
colors = ["#001d43", "#cacccc"]

# Start plotting
plt.figure(figsize=(9, 6))
wedges, texts, autotexts = plt.pie(sizes, labels=labels, colors=colors, 
        autopct='%1.1f%%', # Add numbers to the pie chart
        startangle=90,
        textprops={'fontsize': 20, 'fontweight': 'bold'},
        explode=(0.08, 0),  # Clearer separation
        wedgeprops={'edgecolor': 'white', 'linewidth': 3})  # White borders

# Style the percentage text
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(20)

plt.title("Flight Distribution: Major Hubs vs Other Cities (2021)", 
          size=25, fontweight='bold')

plt.tight_layout()

# Save as svg
#plt.savefig("./Figures/pie_chart_market_share.svg", format="svg")
plt.show()


# Additional Analysis

### Did the 2021 Promotion work?

In [ ]:
df_customer_copy.pivot_table(index=['LoyaltyStatus', 'EnrollmentType'], values=['Total_Flights', 'Subscription_Duration_Days','Flights_in_Subscription', 'Total_Points_Redeemed', "PRR"], aggfunc='median').round(2).sort_values("Total_Flights", ascending=False)

The promotion programms from 2021 didn't really work. The number of flights in subsciption in 1/3 of the normal number for each Loyalty Status, median subscription duration is 900 days less. Point Redemption Rate is similar with the exeption of Aurora which has the highest redemption rate while having the biggest number of flights for the promotion cohort.  

### Who is the most profitable customer group?
Assumptions: <br>
Profitability is defined by the highest number of flights while having the lowest number of points redemed. 
This metric is used as flying a lot without redeeming points means flights are paid in cash and generated revenue. Overall customers who are hording points are good customers for the airline as the bring in new cash instead of spending their points. 

In [ ]:
#print(df_customer_copy['Province or State'].value_counts()) # To check how many customers we have from each province and state.
df_customer_copy.groupby(['Province or State'])[['Income','Subscription_Duration_Days','Total_Flights','percentage_flights_as_sub','mean_spent','distance_airport','Total_Distance_KM', 'Total_Points_Redeemed', "PRR",'Avg_Flight_Dist_KM','Comp_Ratio']].median().round(2).sort_values(["Total_Flights", "PRR"], ascending=[False, True])

The table shows us there are regions with more profitable customers, however the total customer base is still very similar.